# LexiScan Auto - Legal Entity Extractor
This notebook contains the complete project pipeline (OCR, Custom NER, Rules Engine, and API) designed to run easily on Google Colab.

## 1. Environment Setup
Install dependencies for OCR, ML, and API.

In [ ]:
!apt-get update
!apt-get install -y tesseract-ocr poppler-utils
!pip install pytesseract pdf2image spacy fastapi uvicorn python-multipart python-dateutil
!python -m spacy download en_core_web_sm
!pip install nest_asyncio

## 2. OCR Pipeline (`ocr_pipeline.py`)

In [ ]:
import os
import pytesseract 
from pdf2image import convert_from_path
import re

class OCRPipeline:
    def __init__(self, tesseract_cmd=None):
        """Initialize the OCR pipeline."""
        if tesseract_cmd:
            pytesseract.pytesseract.tesseract_cmd = tesseract_cmd
        elif os.name == 'nt':
            # Default Windows tesseract location
            tesseract_path = r'C:\Program Files\Tesseract-OCR\tesseract.exe'
            if os.path.exists(tesseract_path):
                pytesseract.pytesseract.tesseract_cmd = tesseract_path

    def process_pdf(self, pdf_path):
        """Convert PDF to images and extract text using OCR."""
        print(f"Processing PDF: {pdf_path}")
        try:
            images = convert_from_path(pdf_path)
            full_text = ""
            for i, image in enumerate(images):
                print(f"Running OCR on page {i + 1}...")
                text = pytesseract.image_to_string(image)
                full_text += text + "\n"
            return self.clean_text(full_text)
        except Exception as e:
            print(f"Error processing PDF: {e}")
            return ""

    def process_image(self, image_path):
        """Extract text from a single image using OCR."""
        print(f"Processing Image: {image_path}")
        try:
            text = pytesseract.image_to_string(image_path)
            return self.clean_text(text)
        except Exception as e:
            print(f"Error processing Image: {e}")
            return ""
            
    def clean_text(self, text):
        """Basic text cleaning."""
        # Replace multiple spaces with a single space
        text = re.sub(r'[ \t]+', ' ', text)
        # Remove empty lines
        text = re.sub(r'\n\s*\n', '\n', text)
        return text.strip()

if __name__ == "__main__":
    print("OCR Pipeline Initialized successfully.")


## 3. NER Training Script (`train_ner.py`)

In [ ]:
import spacy
from spacy.training.example import Example
import random
import os

# Sample labeled data in Spacy format for demonstration purposes
TRAINING_DATA = [
    ("The contract is effective as of January 1, 2025.", {"entities": [(34, 49, "EFFECTIVE_DATE")]}),
    ("This agreement is made between InfoTact Solutions and Acme Corp.", {"entities": [(31, 49, "PARTY"), (54, 63, "PARTY")]}),
    ("The total monetary value is $50,000 for the services.", {"entities": [(28, 35, "MONEY")]}),
    ("Termination date is set for December 31, 2026.", {"entities": [(28, 45, "TERMINATION_DATE")]}),
]

def train_ner_model(model_name=None, output_dir="models/ner_model", n_iter=10):
    """Trains a custom NER model."""
    print("Setting up NER training pipeline...")
    
    if model_name is not None and spacy.util.is_package(model_name):
        nlp = spacy.load(model_name)
        print(f"Loaded model '{model_name}'")
    else:
        nlp = spacy.blank("en")
        print("Created blank 'en' model")

    if "ner" not in nlp.pipe_names:
        ner = nlp.add_pipe("ner", last=True)
    else:
        ner = nlp.get_pipe("ner")

    # Add labels
    for _, annotations in TRAINING_DATA:
        for ent in annotations.get("entities"):
            ner.add_label(ent[2])

    # Disable other pipes during training
    pipe_exceptions = ["ner", "trf_wordpiecer", "trf_tok2vec"]
    other_pipes = [pipe for pipe in nlp.pipe_names if pipe not in pipe_exceptions]
    
    print("Beginning training...")
    with nlp.disable_pipes(*other_pipes):
        if model_name is None:
            optimizer = nlp.begin_training()
        else:
            optimizer = nlp.resume_training()
            
        for itn in range(n_iter):
            random.shuffle(TRAINING_DATA)
            losses = {}
            for text, annotations in TRAINING_DATA:
                doc = nlp.make_doc(text)
                example = Example.from_dict(doc, annotations)
                nlp.update([example], drop=0.5, sgd=optimizer, losses=losses)
            print(f"Iteration {itn + 1} Losses: {losses}")

    os.makedirs(output_dir, exist_ok=True)
    nlp.to_disk(output_dir)
    print(f"Model saved to {output_dir}")

if __name__ == "__main__":
    train_ner_model()


## 4. NER Inference Pipeline (`ner_model.py`)

In [ ]:
import spacy
import os

class NERPipeline:
    def __init__(self, model_dir="models/ner_model"):
        """Initialize the NER model from the standard directory. Fallback to basic spacy model."""
        if os.path.exists(model_dir):
            self.nlp = spacy.load(model_dir)
        else:
            print("Warning: Custom model not found. Using blank model. Run train_ner.py first.")
            self.nlp = spacy.blank("en")

    def extract_entities(self, text):
        """Extract entities from the provided text."""
        doc = self.nlp(text)
        entities = {}
        for ent in doc.ents:
            if ent.label_ not in entities:
                entities[ent.label_] = []
            entities[ent.label_].append(ent.text)
        return entities

if __name__ == "__main__":
    ner = NERPipeline()
    print("NER Pipeline loaded.")


## 5. Rules Engine (`rules_engine.py`)

In [ ]:
from dateutil import parser
import re

class RulesEngine:
    def __init__(self):
        pass

    def standardize_date(self, date_string):
        """Standardize date string to ISO 8601 format (YYYY-MM-DD)."""
        try:
            parsed_date = parser.parse(date_string, fuzzy=True)
            return parsed_date.strftime("%Y-%m-%d")
        except ValueError:
            return None

    def validate_entities(self, extracted_entities):
        """
        Validate entities logically.
        For example: Termination date cannot precede Effective Date.
        """
        effective_date_raw = extracted_entities.get("EFFECTIVE_DATE")
        termination_date_raw = extracted_entities.get("TERMINATION_DATE")

        if isinstance(effective_date_raw, list) and len(effective_date_raw) > 0:
            effective_date_raw = effective_date_raw[0]
        if isinstance(termination_date_raw, list) and len(termination_date_raw) > 0:
            termination_date_raw = termination_date_raw[0]

        effective_date = self.standardize_date(effective_date_raw) if effective_date_raw else None
        termination_date = self.standardize_date(termination_date_raw) if termination_date_raw else None

        # Clean entities back into standard formats
        cleaned_entities = extracted_entities.copy()
        if effective_date:
            cleaned_entities["EFFECTIVE_DATE_ISO"] = effective_date
        if termination_date:
            cleaned_entities["TERMINATION_DATE_ISO"] = termination_date

        # Constraint checking
        if effective_date and termination_date:
            if effective_date > termination_date:
                cleaned_entities["WARNING"] = "Termination Date precedes Effective Date"

        # Standardizing Money Format
        monetary_value = extracted_entities.get("MONEY")
        if isinstance(monetary_value, list) and len(monetary_value) > 0:
            monetary_value = monetary_value[0]
        if monetary_value:
            cleaned_money = self.standardize_money(monetary_value)
            cleaned_entities["MONEY_CLEANED"] = cleaned_money

        return cleaned_entities
        
    def standardize_money(self, money_str):
        """Remove commas and redundant signs from money amounts."""
        cleaned = re.sub(r'[^\d.]', '', money_str)
        return cleaned

if __name__ == "__main__":
    re_engine = RulesEngine()
    print("Rules Engine Initialized.")


## 6. FastAPI Service (`api.py`)

In [ ]:
from fastapi import FastAPI, File, UploadFile, HTTPException
from ocr_pipeline import OCRPipeline
from ner_model import NERPipeline
from rules_engine import RulesEngine
import os
import shutil

app = FastAPI(title="LexiScan Auto API")

# Initialize models and pipelines
ocr = OCRPipeline()
ner = NERPipeline()
rules = RulesEngine()

@app.post("/extract")
async def extract_entities(file: UploadFile = File(...)):
    if not file.filename.lower().endswith(('.pdf', '.png', '.jpg', '.jpeg')):
        raise HTTPException(status_code=400, detail="Only PDF and Image files are supported.")
        
    temp_file_path = f"temp_{file.filename}"
    try:
        with open(temp_file_path, "wb") as buffer:
            shutil.copyfileobj(file.file, buffer)
            
        # Step 1: OCR Pipeline
        if file.filename.lower().endswith('.pdf'):
            text = ocr.process_pdf(temp_file_path)
        else:
            text = ocr.process_image(temp_file_path)
        
        # Step 2: NER Model
        raw_entities = ner.extract_entities(text)
        
        # Step 3: Rules Engine
        structured_output = rules.validate_entities(raw_entities)
        
        return {
            "status": "success",
            "extracted_text_preview": text[:200] + "..." if len(text) > 200 else text,
            "entities": structured_output
        }
        
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))
    finally:
        if os.path.exists(temp_file_path):
            os.remove(temp_file_path)

if __name__ == "__main__":
    import uvicorn
    #uvicorn.run(app, host="0.0.0.0", port=8001)


## 7. Run Local Test Pipeline
Run the NER training initially, then test the text pipeline natively in Colab.

In [ ]:
# 1. Train model locally in colab
train_ner_model()

# 2. Simulate pipeline
print("\n--- Pipeline Initialized ---")
ocr = OCRPipeline()
ner = NERPipeline()
rules = RulesEngine()

text = "This contract is effective as of January 1, 2025. The termination date is December 31, 2026. The total cost is $50,000 for Acme Corp."
print("\nInput Text:", text)

entities = ner.extract_entities(text)
final_output = rules.validate_entities(entities)

print("\nExtracted and Validated JSON:")
import json
print(json.dumps(final_output, indent=4))


## 8. Run API Server using ngrok/localtunnel (Optional for Colab)

In [ ]:
import nest_asyncio
import uvicorn
from multiprocessing import Process
import time

# nest_asyncio.apply()
# If you want to tunnel the port out of colab, you can use localtunnel:
# !npm install -g localtunnel
# get_ipython().system_raw('lt --port 8080 >> lt_url.txt 2>&1 &')
# uvicorn.run(app, host="0.0.0.0", port=8080)
print("Uncomment instructions above to bind API in Colab.")
